In [1]:
import pandas as pd
data=pd.read_csv(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Dataset\Processed_data.csv")
display(data.head(5))

,Date,Berkeley_Temperature,CRU_Temperature,NASA_Temperature,NOAA_Temperature,Global_Temp_Avg,Warming_Velocity_12m,Warming_Velocity_10y,Temp_Volatility_12m,Trend_Component,Seasonal_Component,Residual_Component,Decade
0,1850-01-01,-0.380,-0.734,NaN,-1.046,-0.720,NaN,NaN,NaN,-0.503663,-0.151874,-0.064463,1850
1,1850-02-01,-0.186,-0.360,NaN,-0.777,-0.441,NaN,NaN,NaN,-0.500248,0.007192,0.052056,1850
2,1850-03-01,-0.220,-0.627,NaN,-0.853,-0.567,NaN,NaN,NaN,-0.496684,-0.128772,0.058456,1850
3,1850-04-01,-0.738,-0.605,NaN,-0.990,-0.778,NaN,NaN,NaN,-0.492934,-0.318902,0.033835,1850
4,1850-05-01,-0.276,-0.532,NaN,-0.839,-0.549,NaN,NaN,NaN,-0.488998,-0.025395,-0.034607,1850


In [2]:
df=data[["Date",
        "Global_Temp_Avg",
         "Warming_Velocity_12m",
         "Warming_Velocity_10y",
         "Temp_Volatility_12m",
         "Trend_Component",
         "Seasonal_Component",
         "Residual_Component"]]
display(df.head(5))

,Date,Global_Temp_Avg,Warming_Velocity_12m,Warming_Velocity_10y,Temp_Volatility_12m,Trend_Component,Seasonal_Component,Residual_Component
0,1850-01-01,-0.720,NaN,NaN,NaN,-0.503663,-0.151874,-0.064463
1,1850-02-01,-0.441,NaN,NaN,NaN,-0.500248,0.007192,0.052056
2,1850-03-01,-0.567,NaN,NaN,NaN,-0.496684,-0.128772,0.058456
3,1850-04-01,-0.778,NaN,NaN,NaN,-0.492934,-0.318902,0.033835
4,1850-05-01,-0.549,NaN,NaN,NaN,-0.488998,-0.025395,-0.034607


In [3]:
print("Before handling missing values:\n",df.isnull().sum())
print(df.shape)
df=df.dropna()
print("\nAfter handling missing values:\n",df.isnull().sum())
print(df.shape)

Before handling missing values:
 Date                      0
Global_Temp_Avg           0
Warming_Velocity_12m     12
Warming_Velocity_10y    120
Temp_Volatility_12m      11
Trend_Component           0
Seasonal_Component        0
Residual_Component        0
dtype: int64
(2112, 8)

After handling missing values:
 Date                    0
Global_Temp_Avg         0
Warming_Velocity_12m    0
Warming_Velocity_10y    0
Temp_Volatility_12m     0
Trend_Component         0
Seasonal_Component      0
Residual_Component      0
dtype: int64
(1992, 8)


In [60]:
df.to_csv(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Dataset\truncated_data.csv")
print("saved successfully")

saved successfully


In [2]:
import pandas as pd
df=pd.read_csv(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Dataset\truncated_data.csv")
display(df.head(5))

,Unnamed: 0,Date,Global_Temp_Avg,Warming_Velocity_12m,Warming_Velocity_10y,Temp_Volatility_12m,Trend_Component,Seasonal_Component,Residual_Component
0,120,1860-01-01,-0.490,0.083,0.230,0.0923,-0.427924,-0.148069,0.085993
1,121,1860-02-01,-0.502,-0.023,-0.061,0.0940,-0.432885,-0.091909,0.022794
2,122,1860-03-01,-0.620,-0.160,-0.053,0.1108,-0.438315,-0.043166,-0.138519
3,123,1860-04-01,-0.429,-0.105,0.349,0.1059,-0.445081,0.047800,-0.031719
4,124,1860-05-01,-0.415,-0.154,0.134,0.0909,-0.453749,0.031699,0.007050


In [3]:
df=df.drop(columns=["Unnamed: 0"])
print("Successfully dropped")

Successfully dropped


In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler

# SEQUENCE SHAPING GENERATOR
# Using sequence_length = 60 (a rolling 5-year monthly window for lookbacks)
sequence_length = 60
target_col = 'Global_Temp_Avg'

# Isolate feature column layout from 7-column matrix
feature_cols = [col for col in df.columns if col not in ['Date', target_col]]

# Scale inputs and targets to prevent gradient explosion in deep layers
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(df[feature_cols])
y_scaled = scaler_y.fit_transform(df[[target_col]])

In [5]:
# Slice structural dimensions into overlapping time windows
X_sequences = []
y_targets = []
timeline_dates = []

for i in range(len(df) - sequence_length):
    X_sequences.append(X_scaled[i : i + sequence_length])
    y_targets.append(y_scaled[i + sequence_length])
    timeline_dates.append(df['Date'].iloc[i + sequence_length])

X_final = np.array(X_sequences)  # Shape becomes: (1932, 60, 6)
y_final = np.array(y_targets)    # Shape becomes: (1932, 1)
input_dimension = X_final.shape[2]

# Chronological Train/Test Split (Preserving chronological integrity)
# Using 85% for training and the last 15% for testing
split_idx = int(len(X_final) * 0.85)
X_train, X_test = X_final[:split_idx], X_final[split_idx:]
y_train, y_test = y_final[:split_idx], y_final[split_idx:]


In [6]:
# HYBRID MODEL ARCHITECTURE
inputs = layers.Input(shape=(sequence_length, input_dimension), name="Climate_Input_Window")

# --- BRANCH A: TRANSFORMER LAYER (Long-Term Dependencies) ---
# Map structural metrics into attention hidden space
transformer_projection = layers.Dense(64)(inputs)

# Explicit Positional Signaling (Informs the transformer of temporal ordering)
positions = tf.range(start=0, limit=sequence_length, delta=1)
position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=64)(positions)
transformer_embeddings = transformer_projection + position_embeddings

# Multi-Head Attention Processing
attention_vector = layers.MultiHeadAttention(num_heads=4, key_dim=64)(transformer_embeddings, transformer_embeddings)
attention_vector = layers.Dropout(0.1)(attention_vector)
normalized_attention = layers.LayerNormalization(epsilon=1e-6)(transformer_embeddings + attention_vector)

# Feed-Forward Dense Processing Block
ffn_dense = layers.Dense(128, activation="relu")(normalized_attention)
ffn_dense = layers.Dense(64)(ffn_dense)
transformer_block_out = layers.LayerNormalization(epsilon=1e-6)(normalized_attention + ffn_dense)

# Global Temporal Feature Extraction Pooling
transformer_branch = layers.GlobalAveragePooling1D(name="Transformer_Feature_Extraction")(transformer_block_out)


In [7]:
# --- BRANCH B: LSTM LAYER (Short-Term Sequential Dependencies) ---
lstm_branch = layers.LSTM(64, return_sequences=False, name="LSTM_Feature_Extraction")(inputs)

# --- DEEP FUSION ENGINE & OUTPUT REGRESSION ---
fused_latent_space = layers.Concatenate()([transformer_branch, lstm_branch])
dense_fusion = layers.Dense(32, activation="relu")(fused_latent_space)
prediction_output = layers.Dense(1, name="Global_Anomaly_Prediction")(dense_fusion)

# Model Instantiation
hybrid_climate_model = Model(inputs=inputs, outputs=prediction_output, name="Hybrid_Climate_Framework")

# Compile Framework
hybrid_climate_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train Model
history = hybrid_climate_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    verbose=1
)

# Evaluate Model
test_loss, test_mae = hybrid_climate_model.evaluate(X_test, y_test, verbose=0)

# Display Metrics
print("\nModel Evaluation Metrics")
print(f"MSE (Mean Squared Error): {test_loss:.4f}")
print(f"MAE (Mean Absolute Error): {test_mae:.4f}")

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - loss: 0.1438 - mae: 0.2777 - val_loss: 0.2199 - val_mae: 0.3482
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - loss: 0.0654 - mae: 0.1985 - val_loss: 0.3357 - val_mae: 0.4558
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 91ms/step - loss: 0.0582 - mae: 0.1884 - val_loss: 0.4123 - val_mae: 0.5162
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - loss: 0.0515 - mae: 0.1751 - val_loss: 0.3145 - val_mae: 0.4291
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 0.0488 - mae: 0.1703 - val_loss: 0.2981 - val_mae: 0.4245
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - loss: 0.0478 - mae: 0.1693 - val_loss: 0.4872 - val_mae: 0.5631
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 118ms/step - loss: 0.0467 - mae: 0.1683 - val_loss: 0.4495 - val_mae: 0.5407
Epoch 8/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - loss: 0.0464 - mae: 0.1677 - val_loss: 0.6363 - val_mae: 0.6768
Epoch 9/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 

In [58]:
# Save the trained weights and architecture
hybrid_climate_model.save(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Codes\hybrid_climate_model.keras")

# Save the fitted Scikit-Learn scalers using pickle
import pickle

with open(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Codes\scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)

with open(r"D:\BCA & MSC Core Notes\MSC\MSC 2ND YEAR\MSC Major project\Codes\scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)

print("All modeling components successfully exported! Ready for Streamlit UI.")

All modeling components successfully exported! Ready for Streamlit UI.
